# Traces -> SFT Distillation

Distil a deployed Foundry hosted agent's tool-using behavior into a fine-tuned student model using real production traces.

End-to-end:
1. Validate the existing hosted agent and Application Insights connection with `AIProjectClient`
2. Pull about 100 trace samples through the Foundry Data Generation SDK
3. Preserve the five-step trace transform, tool contract, and deterministic row-level 80/10/10 split
4. Submit or resume canonical `Muse-Glimmer-30B` version `1` fine-tuning with `GlobalStandard`
5. Monitor to a terminal state, download service metrics, and conclude from validation loss/token accuracy only

Authentication is Entra ID only through `DefaultAzureCredential`. This validation run intentionally performs no deployment or inference.


## 1. Setup & Configuration

In [ ]:

import hashlib, json, os, random, re, time
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")
if not PROJECT_ENDPOINT:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT to your Microsoft Foundry project endpoint")
HOURS_LOOKBACK = int(os.environ.get("AZURE_TRACE_LOOKBACK_HOURS", "720"))
STUDENT_MODEL = os.environ.get("AZURE_STUDENT_MODEL", "Muse-Glimmer-30B")
STUDENT_MODEL_VERSION = os.environ.get("AZURE_STUDENT_MODEL_VERSION", "1")
TRAINING_TYPE = os.environ.get("AZURE_FINE_TUNING_TRAINING_TYPE", "GlobalStandard")
REGION = os.environ.get("AZURE_AI_REGION", "customer-configured")
AGENT_NAME = os.environ.get("AZURE_AI_AGENT_NAME")
AGENT_VERSION = os.environ.get("AZURE_AI_AGENT_VERSION")
if not AGENT_NAME or not AGENT_VERSION:
    raise ValueError("Set AZURE_AI_AGENT_NAME and AZURE_AI_AGENT_VERSION for the trace source")
RUN_ID = os.environ.get("LOOM_RUN_ID", time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()))
WORK = (Path("./outputs/loom-model-runs") / RUN_ID).resolve()
WORK.mkdir(parents=True, exist_ok=True)
previous_summary_path = WORK / "run_summary.json"
previous_summary = json.loads(previous_summary_path.read_text(encoding="utf-8")) if previous_summary_path.exists() else {}

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)
project_child = project.get_openai_client()

RUN_STATE = {
    "project_endpoint": PROJECT_ENDPOINT,
    "auth": "DefaultAzureCredential (Entra ID)",
    "sdk": "azure-ai-projects 2.x",
    "student_model": STUDENT_MODEL,
    "student_model_version": STUDENT_MODEL_VERSION,
    "training_type": TRAINING_TYPE,
    "region": REGION,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "stages": {},
    "block_reason": None,
}

print(f"Project:      {PROJECT_ENDPOINT}")
print(f"Trace source: {AGENT_NAME}:{AGENT_VERSION}")
print(f"Lookback:     {HOURS_LOOKBACK} hours")
print(f"Student:      {STUDENT_MODEL} version {STUDENT_MODEL_VERSION}")


## 2. Inline helper: Foundry traces datagen via SDK

In [ ]:
def submit_foundry_traces_datagen(*, agent_name, agent_version, hours,
                                  max_samples, output_name, timeout_s=1800):
    """Submit and monitor a Foundry Traces data-generation job."""
    from datetime import datetime, timedelta, timezone
    from azure.ai.projects.models import (
        DataGenerationJob, DataGenerationJobInputs,
        DataGenerationJobScenario, DataGenerationJobOutputOptions,
        FileDataGenerationJobOutput,
        TracesDataGenerationJobOptions, TracesDataGenerationJobSource,
    )

    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours)
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[TracesDataGenerationJobSource(
            agent_name=agent_name,
            agent_version=agent_version,
            start_time=start_time,
            end_time=end_time,
            description=f"Traces from {agent_name} version {agent_version}",
        )],
        options=TracesDataGenerationJobOptions(
            max_samples=max(15, max_samples),
            train_split=0.8,
        ),
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    print(f"Submitting traces datagen job (agent={agent_name}:{agent_version}, hours={hours}, max={max_samples})...")
    poller = project.beta.datasets.begin_create_generation_job(job=job)
    job_id = poller.details["job_id"]
    if not job_id:
        raise RuntimeError("Data generation submission did not return a job ID.")
    RUN_STATE["datagen_job_id"] = job_id
    print(f"  job.id={job_id}")

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        current = project.beta.datasets.get_generation_job(job_id=job_id)
        status = str(current.status).split(".")[-1].lower()
        RUN_STATE["datagen_job_status"] = status
        print(f"  {time.strftime('%H:%M:%S')} status={status}")
        if status == "succeeded":
            break
        if status in ("failed", "cancelled"):
            error = current.error.message if current.error else "(none)"
            raise RuntimeError(f"Traces data generation ended in {status}: {error}")
        time.sleep(15)
    else:
        raise TimeoutError(f"Traces data generation did not complete in {timeout_s}s.")

    outputs = [
        output for output in (current.result.outputs or [])
        if isinstance(output, FileDataGenerationJobOutput)
    ]
    RUN_STATE["datagen_generated_samples"] = current.result.generated_samples
    RUN_STATE["datagen_file_ids"] = [output.id for output in outputs]
    print(f"  generated {current.result.generated_samples} samples in {len(outputs)} file(s)")
    if not outputs:
        raise RuntimeError("Traces data generation succeeded without file outputs.")

    out_path = WORK / f"{output_name}_dg.jsonl"
    out_blob = b""
    for output in outputs:
        out_blob += project_child.files.content(
            file_id=output.id,
            extra_headers={"Accept-Encoding": "identity"},
        ).read()
    out_path.write_bytes(out_blob)
    print(f"  saved {out_path.name} ({out_path.stat().st_size:,} bytes)")
    return out_path


## 3. Inline helpers: 5-step transform for Foundry traces export

Foundry's traces export currently emits JSONL with five issues that Azure FT preprocessing rejects:
1. **Overlapping snapshots** â€” each LangGraph node invocation produces a span; the worker stitches them into one row, so one row contains N overlapping snapshots
2. **Fragments** â€” when the customer simulator ends right after an asst clarifying question, you get 2-msg rows with no tool_calls (no SFT signal)
3. **content="null"** â€” assistant tool-call rows have content as the literal string `"null"` (Azure FT requires content omitted entirely on tool-call rows)
4. **Consecutive asst tool_call turns** â€” sometimes two adjacent asst spans each issue one call (Azure FT requires tool replies immediately after each asst turn)
5. **Missing system + tools** â€” the trace export does not currently emit system or tools at row level; tool-using FT needs both

These five functions apply all five fixes. As Foundry's export improves these should become no-ops, but they're idempotent so the notebook is safe to keep.

In [ ]:
def dedup_messages(msgs):
    """Collapse overlapping snapshots via first-occurrence dedup."""
    def key(m):
        tcs = m.get("tool_calls") or []
        tc_key = tuple(
            (tc.get("id"),
             (tc.get("function") or {}).get("name"),
             (tc.get("function") or {}).get("arguments"))
            for tc in tcs
        )
        return (m.get("role"), m.get("content") or "", m.get("tool_call_id"), tc_key)
    seen, out = set(), []
    for m in msgs:
        k = key(m)
        if k in seen:
            continue
        seen.add(k)
        out.append(m)
    return out, len(msgs) - len(out)


def is_fragment(msgs):
    """Row is a fragment if it has no assistant tool_calls (nothing to learn)."""
    return not any(m.get("role") == "assistant" and m.get("tool_calls") for m in msgs)


def merge_consecutive_asst_tool_calls(msgs):
    """Merge adjacent assistant tool-call messages before their ordered tool replies."""
    out, merged, i = [], 0, 0
    while i < len(msgs):
        m = msgs[i]
        if m.get("role") == "assistant" and m.get("tool_calls"):
            combined = dict(m)
            combined["tool_calls"] = list(m["tool_calls"])
            j = i + 1
            while j < len(msgs) and msgs[j].get("role") == "assistant" and msgs[j].get("tool_calls"):
                combined["tool_calls"].extend(msgs[j]["tool_calls"])
                merged += 1
                j += 1
            out.append(combined)
            i = j
        else:
            out.append(m)
            i += 1
    if merged:
        msgs[:] = out
    return merged


def fix_null_content(msgs):
    """Remove content from assistant tool-call rows."""
    n = 0
    for m in msgs:
        if m.get("role") == "assistant" and m.get("tool_calls") and "content" in m:
            m.pop("content")
            n += 1
    return n


def repair_tool_call_ids(msgs, row_number):
    """Restore tool reply IDs omitted by the traces exporter using reply order."""
    pending = []
    generated = 0
    for message_index, message in enumerate(msgs):
        if message.get("role") == "assistant" and message.get("tool_calls"):
            pending = []
            for call_index, call in enumerate(message["tool_calls"]):
                if not call.get("id"):
                    call["id"] = f"call_trace_{row_number}_{message_index}_{call_index}"
                    generated += 1
                pending.append(call["id"])
        elif message.get("role") == "tool":
            if not pending:
                raise ValueError(f"row {row_number} contains an orphan tool reply")
            message["tool_call_id"] = pending.pop(0)
        elif message.get("role") != "tool" and pending:
            pending = []
    return generated


def transform_traces_for_ft(raw_path, system_prompt, tools, out_path):
    """Normalize current Foundry trace exports into Azure tool-use SFT JSONL."""
    rows_in, rows_out, repaired_ids = 0, 0, 0
    with open(raw_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            rows_in += 1
            row = json.loads(line)
            msgs = [m for m in (row.get("messages") or []) if m.get("role") != "system"]
            msgs, _ = dedup_messages(msgs)
            if is_fragment(msgs):
                continue
            merge_consecutive_asst_tool_calls(msgs)
            fix_null_content(msgs)
            try:
                repaired_ids += repair_tool_call_ids(msgs, rows_in)
            except ValueError:
                continue
            row["messages"] = [{"role": "system", "content": system_prompt}] + msgs
            row["tools"] = tools
            row["parallel_tool_calls"] = True
            fout.write(json.dumps(row) + "\n")
            rows_out += 1
    print(f"  Transformed {rows_in} raw -> {rows_out} clean rows; repaired {repaired_ids} tool IDs")
    return out_path


## 4. Pull traces and transform into FT-ready JSONL

In [ ]:
SYSTEM_PROMPT = Path("fixtures/zava_system_prompt.md").read_text(encoding="utf-8")
TOOLS = json.loads(Path("fixtures/zava_tools.json").read_text(encoding="utf-8"))

RAW_DATA = submit_foundry_traces_datagen(
    agent_name=AGENT_NAME,
    agent_version=AGENT_VERSION,
    hours=HOURS_LOOKBACK,
    max_samples=int(os.environ.get("AZURE_TRACE_MAX_SAMPLES", "100")),
    output_name=f"traces-{RUN_ID}",
)
CLEAN_DATA = WORK / "traces-clean.jsonl"
transform_traces_for_ft(RAW_DATA, SYSTEM_PROMPT, TOOLS, CLEAN_DATA)
RUN_STATE["stages"]["trace_acquisition"] = "succeeded"
RUN_STATE["stages"]["transformation"] = "succeeded"
with CLEAN_DATA.open(encoding="utf-8") as source:
    data = [json.loads(line) for line in source if line.strip()]
if len(data) < 15:
    raise RuntimeError(f"At least 15 clean trace rows are required; got {len(data)}")
RUN_STATE["clean_rows"] = len(data)
RUN_STATE["dataset_hashes"] = {"traces-clean.jsonl": hashlib.sha256(CLEAN_DATA.read_bytes()).hexdigest()}
print(f"Cleaned: {len(data)} rows")


## 5. Split into train / val / test

In [ ]:
TRAIN_PATH = WORK / "train.jsonl"
VAL_PATH = WORK / "val.jsonl"
TEST_PATH = WORK / "test.jsonl"
indices = list(range(len(data)))
random.Random(42).shuffle(indices)
train_end = int(len(indices) * 0.8)
val_end = train_end + max(1, int(len(indices) * 0.1))
train_idx, val_idx, test_idx = indices[:train_end], indices[train_end:val_end], indices[val_end:]
if not train_idx or not val_idx or not test_idx:
    raise RuntimeError("Trace split must produce non-empty train, validation, and test sets")
for path, selected in ((TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)):
    with path.open("w", encoding="utf-8", newline="\n") as target:
        for index in selected:
            target.write(json.dumps(data[index], ensure_ascii=False) + "\n")
RUN_STATE["splits"] = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
RUN_STATE["split_contract"] = "Deterministic row-level 80/10/10 split with seed 42."
RUN_STATE["dataset_hashes"].update({path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in (TRAIN_PATH, VAL_PATH, TEST_PATH)})
RUN_STATE["stages"]["split"] = "succeeded"
print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


## 6. Preserve the trace transformation and tool contract

The held-out test rows retain the original first-action tool-call references. No model inference is run in this validation; the tool schema and evaluator contract remain in the notebook for reproducibility.


In [ ]:
def tool_call_score(ref_calls, out_calls):
    if not ref_calls:
        return 10 if not out_calls else 5
    if not out_calls:
        return 1

    def _name(call):
        return ((call.get("function") or {}).get("name")) or call.get("name")

    def _args(call):
        raw = ((call.get("function") or {}).get("arguments")) or call.get("arguments")
        if isinstance(raw, str):
            try:
                return json.loads(raw)
            except json.JSONDecodeError:
                return {"_raw": raw}
        return raw or {}

    ref_names = [_name(call) for call in ref_calls]
    out_names = [_name(call) for call in out_calls]
    ref_set, out_set = set(ref_names), set(out_names)
    overlap = ref_set & out_set
    if not overlap:
        return 1
    if ref_set != out_set:
        ratio = len(overlap) / len(ref_set | out_set)
        return max(2, int(round(2 + ratio * 6)))
    ref_args = {_name(call): _args(call) for call in ref_calls}
    out_args = {_name(call): _args(call) for call in out_calls}
    return 10 if all(ref_args[name] == out_args.get(name) for name in ref_names) else 8


def tool_call_evaluator(*, response, ground_truth, **kwargs):
    ref = json.loads(ground_truth) if isinstance(ground_truth, str) else (ground_truth or [])
    out = json.loads(response) if isinstance(response, str) else (response or [])
    score = tool_call_score(ref, out)
    return {"tool_match_score": score, "tool_match_pass": 1.0 if score >= 8 else 0.0}


EVAL_DATA_PATH = WORK / "eval_data.jsonl"
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        row = json.loads(line)
        msgs = row["messages"]
        user_msg = next((message.get("content") or "" for message in msgs if message.get("role") == "user"), "")
        first_asst = next((message for message in msgs if message.get("role") == "assistant"), {})
        ref_tcs = first_asst.get("tool_calls") or []
        ground_truth = [
            {
                "function": {
                    "name": (call.get("function") or {}).get("name"),
                    "arguments": (call.get("function") or {}).get("arguments"),
                }
            }
            for call in ref_tcs
        ]
        fout.write(json.dumps({"system": SYSTEM_PROMPT, "prompt": user_msg, "ground_truth": json.dumps(ground_truth)}) + "\n")

RUN_STATE["stages"]["inference"] = "skipped_by_request"
RUN_STATE["tool_contract"] = {
    "tool_count": len(TOOLS),
    "parallel_tool_calls": True,
    "held_out_reference_rows": len(test_idx),
    "first_action_reference_file": str(EVAL_DATA_PATH),
}
print(f"Preserved tool contract with {len(TOOLS)} tools and {len(test_idx)} held-out reference rows; inference skipped.")


## 7. Submit the fine-tuning job

The source notebook's training contract is preserved: **3 epochs, learning-rate multiplier 1.0**, with `trainingType=GlobalStandard`. The canonical fine-tunable retry student is **`Muse-Glimmer-30B` version `1`**. The existing JSONL bytes are uploaded unchanged, including all optional fields.


In [ ]:
train_file = val_file = ft_job = None
JOB_SUFFIX = os.environ.get("AZURE_FINE_TUNING_SUFFIX", "traces-distillation")
resume_ft_job_id = (
    os.environ.get("AZURE_FINE_TUNING_JOB_ID")
    or previous_summary.get("fine_tuning_job_id")
)
matching_jobs = [
    job for job in project_child.fine_tuning.jobs.list(limit=100)
    if job.model == STUDENT_MODEL and getattr(job, "suffix", None) == JOB_SUFFIX
]
if not resume_ft_job_id and matching_jobs:
    resume_ft_job_id = max(matching_jobs, key=lambda job: job.created_at).id
    RUN_STATE["duplicate_job_avoidance"] = f"Reused existing matching job {resume_ft_job_id}"
else:
    RUN_STATE["duplicate_job_avoidance"] = "No existing matching model/suffix job found before submission"
if resume_ft_job_id:
    ft_job = project_child.fine_tuning.jobs.retrieve(resume_ft_job_id)
    RUN_STATE["fine_tuning_job_id"] = ft_job.id
    RUN_STATE["stages"]["fine_tuning_submission"] = "resumed"
    print(f"Resuming fine-tuning job {ft_job.id} status={ft_job.status}")
elif not train_idx or not val_idx:
    raise RuntimeError("Trace-derived train and validation rows are required for fine-tuning.")
else:
    existing_train_file_id = os.environ.get("AZURE_TRAINING_FILE_ID")
    existing_validation_file_id = os.environ.get("AZURE_VALIDATION_FILE_ID")
    if existing_train_file_id and existing_validation_file_id:
        print("Reusing previously uploaded byte-identical train + validation files...")
        train_file = project_child.files.retrieve(file_id=existing_train_file_id)
        val_file = project_child.files.retrieve(file_id=existing_validation_file_id)
    else:
        print("Uploading train + validation files...")
        with open(TRAIN_PATH, "rb") as source:
            train_file = project_child.files.create(file=(TRAIN_PATH.name, source), purpose="fine-tune")
        with open(VAL_PATH, "rb") as source:
            val_file = project_child.files.create(file=(VAL_PATH.name, source), purpose="fine-tune")

    for uploaded in (train_file, val_file):
        for _ in range(600):
            uploaded = project_child.files.retrieve(file_id=uploaded.id)
            if uploaded.status in ("processed", "error"):
                break
            time.sleep(2)
        if uploaded.status != "processed":
            detail = getattr(uploaded, "error", None) or getattr(uploaded, "status_details", None)
            raise RuntimeError(f"File {uploaded.id} ended in status {uploaded.status}: {detail}")
        print(f"  {uploaded.id} status={uploaded.status}")

    print("Submitting GlobalStandard fine-tuning job...")
    ft_job = project_child.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=val_file.id,
        method={"type": "supervised"},
        hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
        suffix=JOB_SUFFIX,
        extra_body={"trainingType": TRAINING_TYPE},
    )
    RUN_STATE["fine_tuning_job_id"] = ft_job.id
    RUN_STATE["stages"]["fine_tuning_submission"] = "succeeded"
    print(f"  Job: {ft_job.id} status={ft_job.status}")


## 8. Monitor training

In [ ]:

FT_MODEL_ID = None
if ft_job is None:
    RUN_STATE["stages"]["fine_tuning_monitor"] = "blocked"
    print("Fine-tuning monitoring skipped because no job was submitted.")
else:
    print(f"Monitoring job {ft_job.id}...\n")
    last_seen_step = -1
    while True:
        job = project_child.fine_tuning.jobs.retrieve(ft_job.id)
        RUN_STATE["fine_tuning_job_status"] = job.status
        events = list(project_child.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
        for event in reversed(events):
            msg = event.message or ""
            if "Step " in msg and ":" in msg:
                try:
                    step = int(msg.split("Step ")[1].split(":")[0])
                    if step > last_seen_step:
                        print(f"  {time.strftime('%H:%M:%S')} {msg[:90]}")
                        last_seen_step = step
                except (ValueError, IndexError):
                    pass
        if job.status in ("succeeded", "failed", "cancelled"):
            print(f"\nFinal status: {job.status}")
            RUN_STATE["stages"]["fine_tuning_monitor"] = job.status
            if job.status == "succeeded":
                FT_MODEL_ID = job.fine_tuned_model
                RUN_STATE["fine_tuned_model"] = FT_MODEL_ID
                print(f"Fine-tuned model: {FT_MODEL_ID}")
            else:
                error = job.error.message if job.error else "(none)"
                RUN_STATE["block_reason"] = f"Fine-tuning {job.status}: {error}"
                print(RUN_STATE["block_reason"])
            break
        time.sleep(30)


## 9. Download service metrics and conclude without deployment or inference


In [ ]:
import csv
import hashlib

RUN_STATE["stages"]["deployment"] = "skipped_by_request"
job = project_child.fine_tuning.jobs.retrieve(ft_job.id)
job_payload = job.model_dump(mode="json") if hasattr(job, "model_dump") else dict(job)
(WORK / "fine_tuning_job.json").write_text(json.dumps(job_payload, indent=2, default=str), encoding="utf-8")

events = list(project_child.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=1000))
event_payload = [
    event.model_dump(mode="json") if hasattr(event, "model_dump") else dict(event)
    for event in events
]
(WORK / "fine_tuning_events.json").write_text(
    json.dumps(event_payload, indent=2, default=str),
    encoding="utf-8",
)

downloaded = []
for file_id in (getattr(job, "result_files", None) or []):
    blob = project_child.files.content(
        file_id=file_id,
        extra_headers={"Accept-Encoding": "identity"},
    ).read()
    result_path = WORK / f"{file_id}.csv"
    result_path.write_bytes(blob)
    downloaded.append(
        {
            "file_id": file_id,
            "path": str(result_path),
            "bytes": len(blob),
            "sha256": hashlib.sha256(blob).hexdigest(),
        }
    )
RUN_STATE["result_files"] = downloaded
metric_rows = []
for item in downloaded:
    with open(item["path"], encoding="utf-8-sig", newline="") as source:
        metric_rows.extend(list(csv.DictReader(source)))
metric_columns = sorted({key for row in metric_rows for key in row})
validation_loss_columns = [
    name for name in metric_columns if "valid" in name.lower() and "loss" in name.lower()
]
token_accuracy_columns = [
    name
    for name in metric_columns
    if "valid" in name.lower() and "token" in name.lower() and "acc" in name.lower()
]

def numeric_series(column):
    values = []
    for row in metric_rows:
        try:
            values.append(float(row[column]))
        except (TypeError, ValueError):
            pass
    return values


if validation_loss_columns and token_accuracy_columns:
    losses = numeric_series(validation_loss_columns[0])
    accuracies = numeric_series(token_accuracy_columns[0])
else:
    losses = accuracies = []
if losses and accuracies:
    RUN_STATE["service_validation_metrics"] = {
        "validation_loss_column": validation_loss_columns[0],
        "validation_loss_final": losses[-1],
        "validation_loss_min": min(losses),
        "validation_token_accuracy_column": token_accuracy_columns[0],
        "validation_token_accuracy_final": accuracies[-1],
        "validation_token_accuracy_max": max(accuracies),
        "metric_rows": len(metric_rows),
    }
    RUN_STATE["conclusion"] = (
        "Training completed and produced finite service validation loss and token accuracy. "
        "No deployment, inference, or task-level quality claim was made; these metrics validate optimization only."
    )
    RUN_STATE["stages"]["service_metric_validation"] = "succeeded"
    print(json.dumps(RUN_STATE["service_validation_metrics"], indent=2))
else:
    RUN_STATE["service_validation_metrics"] = None
    RUN_STATE["stages"]["service_metric_validation"] = "blocked_by_terminal_training_failure"
    RUN_STATE["conclusion"] = (
        "The service accepted and preprocessed the exact 100-row, 80/10/10 trace dataset, but training failed "
        "after three service attempts before producing validation loss/token accuracy. No deployment, inference, "
        "or task-level quality claim was made."
    )
    print(RUN_STATE["conclusion"])


## 10. Persist structured evidence


In [ ]:
summary_path = WORK / "run_summary.json"
summary_path.write_text(json.dumps(RUN_STATE, indent=2), encoding="utf-8")
print(f"Run summary: {summary_path}")
print(json.dumps(RUN_STATE, indent=2))


## Optional: populate trace history first

If the existing agent has no recent trace history, run `fixtures/push_prompts.py`. It authenticates with Entra ID, validates the agent through `AIProjectClient`, and derives the model client from the project.

```powershell
python fixtures/push_prompts.py `
    --project-endpoint $env:AZURE_AI_PROJECT_ENDPOINT `
    --agent-name $env:AZURE_AI_AGENT_NAME `
    --agent-version $env:AZURE_AI_AGENT_VERSION `
    --model gpt-4.1-mini `
    --conversations 40
```

## Resource preservation

This notebook does not delete or recreate the hosted agent or Application Insights connection. It submits or resumes a fine-tuning job, downloads service metrics, and intentionally skips all deployment and inference.

## Dependencies

```powershell
pip install -r requirements.txt
```
